# 03 — WordCount reading from and writing to Silo (S3A)

Unlike 01/02, this notebook keeps its input and output in **Silo**, the lab's S3-compatible object storage:

```text
Spark (Driver / Executors) ──S3A (hadoop-aws 3.4.2, AWS SDK v2)──► http://silo.spark-lab.svc:9000
                                                                      bucket spark-data
                                                                      ├── shared/wordcount/   public sample input (read)
                                                                      └── users/<user>/        your output (write)
```

**Steps:** configure S3A → list the shared input → read + WordCount → write Parquet to your prefix → list output → read back → check in the Silo Console.

**Where Spark runs:**
- `EXECUTORS = "local"` (default): Driver and tasks run inside this Notebook Pod.
- `EXECUTORS = "k8s"`: Client Mode. Two Executor Pods in your namespace do the S3A reads and writes; the Driver only coordinates.

**Credentials:** `SILO_ENDPOINT`, `SILO_ACCESS_KEY` and `SILO_SECRET_KEY` come from the Secret `silo-shared-credentials` in your namespace: a **non-root** Silo user shared by all lab users, limited to bucket `spark-data`. They never appear in the notebook.

**Not isolation:** every user has the same Silo credentials, so `users/<user>/` is a naming convention, not a security boundary. Deleting your account deletes `users/<user>/`.

> This folder is refreshed from the runtime image each time the Notebook Server starts. Copy the notebook elsewhere under `work/` before editing it.

In [ ]:
EXECUTORS = "local"   # "local" or "k8s"

## 1. Create a SparkSession with the S3A settings

| Setting | Why |
|---|---|
| `fs.s3a.endpoint` | Silo instead of AWS |
| `fs.s3a.path.style.access=true` | `http://silo:9000/spark-data/key`, not `http://spark-data.silo:9000/key` (no wildcard DNS in the cluster) |
| `fs.s3a.connection.ssl.enabled=false` | Silo serves plain HTTP inside the cluster |
| `SimpleAWSCredentialsProvider` | Use the access/secret key given here, not the AWS default credential chain |

`spark.hadoop.*` settings are copied into the Hadoop configuration that the Driver ships to every Executor, so Executor Pods need no extra credentials.

In [ ]:
import os, socket
from pyspark.sql import SparkSession

endpoint = os.environ["SILO_ENDPOINT"]
user = os.environ["SPARK_USER"]

builder = (
    SparkSession.builder
    .appName(f"WordCount-S3A-{EXECUTORS}")
    .config("spark.hadoop.fs.s3a.endpoint", endpoint)
    .config("spark.hadoop.fs.s3a.access.key", os.environ["SILO_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["SILO_SECRET_KEY"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)

if EXECUTORS == "k8s":
    builder = (
        builder
        .master("k8s://https://kubernetes.default.svc:443")
        .config("spark.submit.deployMode", "client")
        .config("spark.kubernetes.namespace", os.environ["SPARK_LAB_NAMESPACE"])
        .config("spark.driver.host", os.environ["SPARK_LAB_DRIVER_HOST"])
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.kubernetes.driver.pod.name", socket.gethostname())
        .config("spark.kubernetes.container.image", os.environ.get("SPARK_LAB_RUNTIME_IMAGE", "spark-lab-runtime:0.3.0"))
        .config("spark.kubernetes.container.image.pullPolicy", "Never")
        .config("spark.executor.instances", "2")
        .config("spark.executor.memory", "1g")
        .config("spark.kubernetes.executor.request.cores", "500m")
    )
else:
    builder = builder.master("local[*]")

spark = builder.getOrCreate()
sc = spark.sparkContext
print("master    :", sc.master)
print("endpoint  :", endpoint)
print(f"Spark UI  : http://localhost:8080/spark-ui/{user}/")

if EXECUTORS == "k8s":
    import time
    for _ in range(60):
        if sc._jsc.sc().getExecutorMemoryStatus().size() - 1 >= 2:
            break
        time.sleep(2)
    print("executors :", sc._jsc.sc().getExecutorMemoryStatus().size() - 1)

## 2. Locate the shared input and your output prefix

`startcluster.sh` uploads a public sample to `s3a://spark-data/shared/wordcount/`. You read it; you write only under `s3a://spark-data/users/<user>/`.

In [ ]:
input_path = "s3a://spark-data/shared/wordcount/"
output_path = f"s3a://spark-data/users/{user}/wordcount-s3a/wordcount.parquet"
print("input :", input_path)
print("output:", output_path)

## 3. List the objects through the Hadoop FileSystem API

This is the same S3A client Spark uses. A "directory" on S3 is only a key prefix; S3A shows it as a directory.

In [ ]:
def ls(path):
    jvm = spark._jvm
    p = jvm.org.apache.hadoop.fs.Path(path)
    fs = p.getFileSystem(sc._jsc.hadoopConfiguration())
    for st in fs.listStatus(p):
        kind = "dir " if st.isDirectory() else "file"
        print(f"  {kind} {st.getLen():>6} B  {st.getPath().getName()}")

print(input_path); ls(input_path)

## 4. Read from Silo and count words

The first job reports which Pod read each input partition from Silo. With `EXECUTORS = "k8s"` these are Executor Pod names, not `jupyter-<user>`.

In [ ]:
df = spark.read.text(input_path)

readers = df.rdd.mapPartitions(lambda rows: [(socket.gethostname(), sum(1 for _ in rows))]).collect()
for host, n in readers:
    print(f"read {n} line(s) from Silo on {host}")

counts = (
    df.rdd.flatMap(lambda row: row.value.split())
    .map(lambda w: (w, 1))
    .reduceByKey(lambda a, b: a + b)
    .toDF(["word", "count"])
)
counts.orderBy("count", ascending=False).show(5)

## 5. Write the result to Silo as Parquet, then list it

In [ ]:
counts.coalesce(1).write.mode("overwrite").parquet(output_path)
print(output_path); ls(output_path)

## 6. Read the Parquet back from Silo

In [ ]:
back = spark.read.parquet(output_path)
print("rows:", back.count())
back.orderBy("word").show(truncate=False)

## 7. Check the objects in the Silo Console

1. Open **http://localhost:8080/silo/** (or `http://<mac-ip>:8080/silo/`). The Console login is the Silo **admin** key, which only the lab administrator has (`.lab/credentials.env` on the Mac). Ask the admin, or skip this step: the listings above come from the same storage.
2. Open bucket **spark-data** → `users` → your user name → `wordcount-s3a/wordcount.parquet/`, and `shared/wordcount/` for the input.
3. You should see the same objects as above: `_SUCCESS` and `part-…` files. On the Mac, `scripts/silo-export.sh spark-data users/<user>/` copies them into `./data/silo-export/`.

Silo stores each object on its PVC in MinIO's own format (a directory with `xl.meta`), so these files never appear as plain `.parquet` files on disk. See README §4.

In [ ]:
spark.stop()   # Client Mode: the Driver also deletes its Executor Pods here